# Antigravity × See-through 自動連携パイプライン
このノートブックを「すべてのセルを実行」すると、Google Driveの特定のフォルダを監視し、画像が置かれた瞬間に自動でSee-through（レイヤー分割＆PSD生成）を実行します。
**※「ランタイムのタイプを変更」からハードウェアアクセラレータを「T4 GPU」などのGPUに設定してください。**

In [ ]:
# 1. Google Driveのマウントとフォルダ準備
from google.colab import drive
import os
%env HF_ENDPOINT=https://hf-mirror.com

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/VTuber_Pipeline'
INPUT_DIR = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
PROCESSING_DIR = os.path.join(BASE_DIR, 'processing')
DONE_DIR = os.path.join(BASE_DIR, 'done')

for d in [INPUT_DIR, OUTPUT_DIR, PROCESSING_DIR, DONE_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Google Driveの準備が完了しました。")

In [ ]:
# 2. See-through環境の構築 (初回のみ時間がかかります)
import os
%cd /content
if not os.path.exists('see-through'):
    !git clone https://github.com/shitagaki-lab/see-through.git
%cd see-through

# 依存ライブラリのインストール
!pip install -r requirements.txt

print("✅ See-through環境の構築が完了しました。")

In [ ]:
# 3. 自動監視ループ（常時稼働）
import os
import time
import shutil
import glob
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

print(f"\n⏳ 監視を開始しました...\nフォルダ: {INPUT_DIR} に画像(.png, .jpg)を入れてください。")

while True:
    # 1. inputフォルダの画像を検索
    extensions = ('*.png', '*.jpg', '*.jpeg')
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
        
    if files:
        for file_path in files:
            filename = os.path.basename(file_path)
            processing_path = os.path.join(PROCESSING_DIR, filename)
            
            # 処理中フォルダへ移動（複数プロセスの衝突回避）
            try:
                shutil.move(file_path, processing_path)
            except Exception as e:
                print(f"ファイルの移動に失敗しました: {e}")
                continue
                
            print(f"\n▶️ 新しい画像を検知しました: {filename}")
            print(f"処理を開始します...")
            
            # 2. See-throughの実行
            # 実行ディレクトリは /content/see-through である必要がある
            os.chdir('/content/see-through')
            
            # 一時出力ディレクトリ
            tmp_out = '/content/tmp_output'
            os.makedirs(tmp_out, exist_ok=True)
            
            # 推論コマンド実行
            cmd = f"PYTHONPATH=. python inference/scripts/inference_psd.py --srcp '{processing_path}' --save_dir '{tmp_out}' --save_to_psd"
            !{cmd}
            
            # 3. 処理結果をoutputへ移動
            name_without_ext = os.path.splitext(filename)[0]
            psd_path = os.path.join(tmp_out, name_without_ext, 'psd', f"{name_without_ext}.psd")
            
            if os.path.exists(psd_path):
                out_psd = os.path.join(OUTPUT_DIR, f"{name_without_ext}.psd")
                shutil.copy(psd_path, out_psd)
                print(f"✅ PSDを生成し、Google Driveに保存しました: {out_psd}")
            else:
                print("❌ PSDの生成に失敗しました。")
                
            # 4. 元画像をdoneフォルダへ移動してクリーンアップ
            shutil.move(processing_path, os.path.join(DONE_DIR, filename))
            
            print("⏳ 次の画像を待機しています...")
            
    time.sleep(5)  # 5秒間隔でポーリング
